# CNN Model for Color Detection at Crosshair Position

This notebook trains a Convolutional Neural Network (CNN) to identify colors at a crosshair location in images. The model learns to focus on the center region of images and classify the color present at that point.

**Objective:** Train a CNN that takes an image with a crosshair marker and predicts the RGB color value at the crosshair position.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import cv2
from tqdm import tqdm

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

KeyboardInterrupt: 

## 1. Data Generation

Generate synthetic training data with images containing random colors and a crosshair at the center that marks the target color.

In [ ]:
def generate_crosshair_image(image_size=128, crosshair_width=3, crosshair_length=20):
    """
    Generate a synthetic image with a random background color and a crosshair at the center.
    
    Args:
        image_size: Size of the square image (default 128x128)
        crosshair_width: Width of the crosshair lines
        crosshair_length: Length of each crosshair arm from center
    
    Returns:
        image: RGB image array (normalized to 0-1)
        target_color: RGB color at the crosshair center (normalized to 0-1)
    """
    # Generate random background color
    background_color = np.random.randint(0, 256, 3) / 255.0
    
    # Create image with background color
    image = np.ones((image_size, image_size, 3)) * background_color
    
    # Define crosshair color (contrasting color for visibility)
    crosshair_color = np.array([1.0, 1.0, 1.0]) if np.mean(background_color) < 0.5 else np.array([0.0, 0.0, 0.0])
    
    # Draw crosshair at center
    center = image_size // 2
    
    # Vertical line
    image[center - crosshair_length:center + crosshair_length, 
          center - crosshair_width:center + crosshair_width] = crosshair_color
    
    # Horizontal line
    image[center - crosshair_width:center + crosshair_width,
          center - crosshair_length:center + crosshair_length] = crosshair_color
    
    # Small circle at center
    radius = 2
    y, x = np.ogrid[:image_size, :image_size]
    mask = (x - center)**2 + (y - center)**2 <= radius**2
    image[mask] = crosshair_color
    
    # The target color is the background color (what the crosshair is pointing to)
    target_color = background_color
    
    return image.astype(np.float32), target_color.astype(np.float32)


# Generate dataset
num_samples = 500
image_size = 128

print(f"Generating {num_samples} training samples...")
X_data = []
y_data = []

for _ in tqdm(range(num_samples)):
    image, target_color = generate_crosshair_image(image_size=image_size)
    X_data.append(image)
    y_data.append(target_color)

X_data = np.array(X_data)
y_data = np.array(y_data)

print(f"Dataset shape - Images: {X_data.shape}, Colors: {y_data.shape}")
print(f"  - X_data: {X_data.shape}, dtype: {X_data.dtype}, range: [{X_data.min():.3f}, {X_data.max():.3f}]")
print(f"  - y_data: {y_data.shape}, dtype: {y_data.dtype}, range: [{y_data.min():.3f}, {y_data.max():.3f}]")

In [ ]:
# Visualize sample training images
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Sample Training Images with Crosshairs', fontsize=14, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    ax.imshow(X_data[idx])
    target_rgb = (y_data[idx] * 255).astype(int)
    ax.set_title(f'RGB({target_rgb[0]}, {target_rgb[1]}, {target_rgb[2]})', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ Sample images visualized!")

## 2. Data Preprocessing & Splitting

In [ ]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

print(f"\n✓ Data split complete!")

## 3. CNN Model Architecture

Design a CNN that extracts features from the image and predicts the RGB color values at the crosshair position.

In [ ]:
def build_color_detection_model(input_shape=(128, 128, 3)):
    """
    Build a CNN model for color detection at crosshair position.
    
    Args:
        input_shape: Shape of input images (height, width, channels)
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Fourth Convolutional Block
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Global Average Pooling
        layers.GlobalAveragePooling2D(),
        
        # Dense layers
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Output layer - predict RGB values (3 outputs)
        layers.Dense(3, activation='sigmoid')  # sigmoid for RGB values in range [0, 1]
    ])
    
    return model


# Build model
model = build_color_detection_model()

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',  # Mean Squared Error for regression
    metrics=['mae']  # Mean Absolute Error as evaluation metric
)

# Display model architecture
print("=" * 60)
print("CNN Model Architecture")
print("=" * 60)
model.summary()
print("=" * 60)

## 4. Train the Model

In [ ]:
# Define callbacks for training
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
print("Starting model training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print("✓ Training complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Model Loss Over Epochs', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Mean Absolute Error', fontsize=12)
axes[1].set_title('Model MAE Over Epochs', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training history visualized!")

## 5. Model Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print("=" * 60)
print("Test Set Performance")
print("=" * 60)
print(f"Test Loss (MSE): {test_loss:.6f}")
print(f"Test MAE: {test_mae:.6f}")
print(f"Test MAE in RGB units: {test_mae * 255:.2f}/255")
print("=" * 60)

# Make predictions on test set
y_pred = model.predict(X_test, verbose=0)

# Calculate additional metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"\nDetailed Metrics:")
print(f"  Mean Squared Error (MSE): {mse:.6f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"  Mean Absolute Error (MAE): {mae:.6f}")
print(f"  Mean Absolute Error (MAE in RGB): {mae * 255:.2f}/255")

# Per-channel metrics
for channel, channel_name in enumerate(['Red', 'Green', 'Blue']):
    channel_mae = mean_absolute_error(y_test[:, channel], y_pred[:, channel])
    print(f"  {channel_name} Channel MAE: {channel_mae:.6f} ({channel_mae * 255:.2f}/255)")

In [ ]:
# Visualize predictions vs ground truth
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Predictions vs Ground Truth (Test Set)', fontsize=14, fontweight='bold')

for idx in range(12):
    row = idx // 4
    col = idx % 4
    ax = axes[row, col]
    
    # Create a side-by-side visualization
    img_height, img_width = X_test[idx].shape[0], X_test[idx].shape[1]
    combined = np.zeros((img_height, img_width * 2 + 10, 3))
    
    # Original image with ground truth
    combined[:, :img_width] = X_test[idx]
    
    # Uniform color showing predicted color
    combined[:, img_width+10:] = y_pred[idx]
    
    ax.imshow(combined)
    
    # Get RGB values
    true_rgb = (y_test[idx] * 255).astype(int)
    pred_rgb = (y_pred[idx] * 255).astype(int)
    
    ax.set_title(f'True: RGB({true_rgb[0]}, {true_rgb[1]}, {true_rgb[2]})\n'
                 f'Pred: RGB({pred_rgb[0]}, {pred_rgb[1]}, {pred_rgb[2]})', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ Predictions visualized!")

In [ ]:
# Visualize error distribution
color_errors = np.abs(y_test - y_pred) * 255  # Convert to 0-255 scale

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall error distribution
ax = axes[0, 0]
ax.hist(np.mean(color_errors, axis=1), bins=30, edgecolor='black', alpha=0.7)
ax.set_xlabel('Mean Absolute Error (0-255)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Mean Color Prediction Errors', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# Per-channel errors
ax = axes[0, 1]
for channel, color, name in zip(range(3), ['red', 'green', 'blue'], ['Red', 'Green', 'Blue']):
    ax.hist(color_errors[:, channel], bins=30, alpha=0.6, label=name, edgecolor='black')
ax.set_xlabel('Absolute Error (0-255)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Per-Channel Error Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Box plot
ax = axes[1, 0]
box_data = [color_errors[:, i] for i in range(3)]
bp = ax.boxplot(box_data, labels=['Red', 'Green', 'Blue'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['red', 'green', 'blue']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Absolute Error (0-255)', fontsize=11)
ax.set_title('Error Distribution by Channel', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Scatter plot: predicted vs ground truth
ax = axes[1, 1]
flattened_true = y_test.flatten() * 255
flattened_pred = y_pred.flatten() * 255
ax.scatter(flattened_true, flattened_pred, alpha=0.5, s=20)
ax.plot([0, 255], [0, 255], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Ground Truth (0-255)', fontsize=11)
ax.set_ylabel('Prediction (0-255)', fontsize=11)
ax.set_title('Predicted vs Ground Truth Colors', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)

plt.tight_layout()
plt.show()

print("✓ Error analysis completed!")

## 6. Save and Load Model

In [ ]:
# Save the trained model
import os

model_dir = 'trained_models'
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'color_detection_cnn.h5')
model.save(model_path)

print(f"✓ Model saved to: {model_path}")
print(f"  File size: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

# Save model in TF SavedModel format (recommended)
savedmodel_path = os.path.join(model_dir, 'color_detection_cnn_savedmodel')
model.save(savedmodel_path)
print(f"✓ Model also saved in SavedModel format: {savedmodel_path}")

## 7. Real-Time Color Detection Demo

Use the trained model to predict colors on new images.

In [ ]:
def predict_color_at_crosshair(image):
    """
    Predict the color at the crosshair position for a single image.
    
    Args:
        image: Image array (H, W, 3) with values in [0, 1]
    
    Returns:
        predicted_color: RGB color as [0-255] values
    """
    # Add batch dimension
    image_batch = np.expand_dims(image, axis=0)
    
    # Predict
    prediction = model.predict(image_batch, verbose=0)[0]
    
    # Convert to 0-255 range
    predicted_color = (prediction * 255).astype(int)
    
    return predicted_color


# Generate and test on some new images
print("=" * 80)
print("REAL-TIME COLOR DETECTION - DEMO")
print("=" * 80)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Real-Time Color Detection at Crosshair Position', fontsize=14, fontweight='bold')

for idx in range(6):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    # Generate a new test image
    test_image, true_color = generate_crosshair_image(image_size=128)
    
    # Predict color
    predicted_rgb = predict_color_at_crosshair(test_image)
    true_rgb = (true_color * 255).astype(int)
    
    # Display
    ax.imshow(test_image)
    
    # Color info display
    info_text = f'True RGB: ({true_rgb[0]}, {true_rgb[1]}, {true_rgb[2]})\n'
    info_text += f'Pred RGB: ({predicted_rgb[0]}, {predicted_rgb[1]}, {predicted_rgb[2]})\n'
    error = np.abs(true_rgb - predicted_rgb).mean()
    info_text += f'Avg Error: {error:.1f}/255'
    
    ax.set_title(info_text, fontsize=10, family='monospace')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ Demo completed!")
print("=" * 80)

## 8. Summary & Next Steps

### Model Performance Summary
- **Architecture**: 4-layer CNN with Batch Normalization and Dropout
- **Output**: 3 values (R, G, B) predicting colors in range [0, 1]
- **Loss Function**: Mean Squared Error (MSE) - suitable for regression
- **Training Strategy**: Early stopping with learning rate reduction

### Key Features
✓ Synthetic data generation with crosshair markers
✓ Comprehensive evaluation metrics (MSE, RMSE, MAE)
✓ Per-channel error analysis
✓ Real-time prediction capability
✓ Model visualization and saving

### Next Steps to Improve the Model
1. **More Training Data**: Generate larger dataset with varied backgrounds and lighting
2. **Data Augmentation**: Add rotation, brightness, contrast adjustments
3. **Real Images**: Train on actual camera feed images with real crosshairs
4. **Ensemble Methods**: Train multiple models and average predictions
5. **Transfer Learning**: Use pre-trained models (ResNet, MobileNet) instead of training from scratch
6. **Hyperparameter Tuning**: Optimize learning rate, batch size, dropout rates
7. **Mobile Optimization**: Convert to TFLite for mobile deployment

### Model Usage
```python
# Load saved model
loaded_model = keras.models.load_model('trained_models/color_detection_cnn.h5')

# Predict on new image
prediction = loaded_model.predict(new_image[np.newaxis, ...])[0]
predicted_color_rgb = (prediction * 255).astype(int)
```